# Train a Welsh chat model on Google Colab (free T4)

This runs the whole **lrl-toolkit** pipeline end to end and gives you a small **Welsh chat model you can talk to** — in about **2 hours** on a free Colab GPU.

**Everything is saved on your Google Drive** (corpus, downloads, checkpoints, the trained model, the run manifest), so if Colab disconnects you just re-run and it **resumes** from where it stopped.

> **First:** Runtime → Change runtime type → **GPU**. This installs `lrl-toolkit` from GitHub `main`, so push your latest code there first.

Run the cells top to bottom.

## 1. Mount Google Drive (first — the run needs it)
Authorise when prompted. The notebook refuses to continue unless Drive is connected, because that is where everything is saved and resumed from.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

assert os.path.isdir('/content/drive/MyDrive'), 'Google Drive did not mount — re-run this cell and authorise.'
DRIVE = '/content/drive/MyDrive/lrl-toolkit'
WORKDIR = f'{DRIVE}/welsh'
os.makedirs(WORKDIR, exist_ok=True)
print('\u2713 Drive connected. Everything is saved under:', WORKDIR)

## 2. Confirm the GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU, then re-run.'

## 3. Install lrl-toolkit
Colab already ships a CUDA build of PyTorch, so this only adds the toolkit + its data/train/eval extras (bitsandbytes for QLoRA, fasttext for language ID, etc.).

In [ ]:
!pip install -q 'lrl-toolkit[data,train,eval] @ git+https://github.com/jwilliamsresearch/lrl-toolkit.git'
!pip install -q fasttext-wheel 2>/dev/null || echo 'fasttext-wheel unavailable here; clean stage will skip language filtering (still runs).'

## 4. Patch fasttext for NumPy 2 (no-op if not needed)

In [ ]:
import numpy as np, pathlib
try:
    import fasttext
    if int(np.__version__.split('.')[0]) >= 2:
        p = pathlib.Path(fasttext.__file__).with_name('FastText.py')
        p.write_text(p.read_text().replace('copy=False', 'copy=None'))
        print('\u2713 patched fasttext for NumPy 2')
    else:
        print('NumPy < 2 — no patch needed')
except ImportError:
    print('fasttext not installed — GlotLID off, clean still runs')

## 5. Write the project config (workdir + caches on Drive)
Sized to finish in ~2 hours on a T4. Everything points at Drive so it persists and resumes.

In [ ]:
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'            # model + dataset downloads cached on Drive
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

CONFIG = f'''name: welsh
language: welsh
base_model: qwen3-1.7b
compute: colab_t4
workdir: {WORKDIR}
ingest:
  sources: [wikipedia, culturax, madlad400, fineweb2]
  max_gb: 0.25
clean:
  lang_id: glotlid
  dedup: minhash
  min_quality: 0.6
tokenizer:
  strategy: extend
  added_tokens: 2000
pretrain:
  method: qlora
  max_steps: 300
  seq_len: 1024
convdata:
  translate: [dolly]
  translate_limit: 200
  translate_backend: nllb
  provider: local
  model: Qwen/Qwen2.5-1.5B-Instruct
  synth:
    provider: local
    model: Qwen/Qwen2.5-1.5B-Instruct
    n: 120
  review: false
finetune:
  method: qlora
  epochs: 1
  max_seq_len: 1024
evaluate:
  benchmarks: [native_cloze, perplexity, belebele, global_mmlu, flores]
  limit: 100
export:
  quantize: [gguf_q4_k_m]
  make_ollama_modelfile: true
  push_to_hub: false
'''
open('welsh-colab.yaml', 'w').write(CONFIG)
print(CONFIG)

## 6. Run the pipeline (resumable)
All 8 stages: ingest \u2192 clean \u2192 tokenizer \u2192 pretrain \u2192 convdata \u2192 finetune \u2192 evaluate \u2192 export.

Each finished stage is recorded in the manifest **on Drive**. If Colab disconnects, just **re-run this cell** — completed stages are skipped and it continues.

In [ ]:
!lrl run -c welsh-colab.yaml

## 7. See the report card & model card

In [ ]:
import json
print('=== EVALUATION (coverage-aware) ===')
rep = json.load(open(f'{WORKDIR}/artifacts/evaluate/report_card.json'))
print(json.dumps(rep.get('summary', {}), indent=2, ensure_ascii=False))
print('\n=== MODEL CARD ===')
print(open(f'{WORKDIR}/artifacts/export/MODEL_CARD.md').read())

## 8. Speak to your Welsh model \ud83d\udde3\ufe0f
Loads the merged chat model from Drive and generates a reply. Edit the prompt and re-run.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

mdir = f'{WORKDIR}/artifacts/export/merged'
tok = AutoTokenizer.from_pretrained(mdir)
model = AutoModelForCausalLM.from_pretrained(mdir, torch_dtype=torch.float16, device_map='cuda')

prompt = 'Helo! Sut wyt ti heddiw?'  # Welsh: 'Hello! How are you today?'
msgs = [{'role': 'user', 'content': prompt}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(ids, max_new_tokens=160, do_sample=True, temperature=0.7, top_p=0.9)
print('You:  ', prompt)
print('Model:', tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))